## Energy Operations Intelligence Platform: ETL Pipeline

### Project Overview

This project implements an Extract, Transform, Load (ETL) pipeline designed to gather, process, and analyze data from various sources within an energy operations ecosystem. The primary goal is to create an intelligence platform that provides insights into plant performance, equipment health, fault events, and operational costs. By integrating diverse datasets, this platform aims to enable proactive monitoring, predictive maintenance, and optimized resource allocation for energy assets.

### Business Problem/Use Case

Modern energy operations generate vast amounts of data from sensors, maintenance systems, financial records, and environmental monitoring. Without a structured approach to data integration and analysis, critical insights can be missed, leading to:

*   **Inefficient Operations:** Suboptimal energy generation, increased downtime, and higher operating costs due to lack of real-time performance visibility.
*   **Equipment Failures:** Unanticipated equipment breakdowns and reduced lifespan due to delayed detection of deteriorating health.
*   **Unplanned Maintenance:** Reactive maintenance strategies instead of proactive interventions, leading to higher costs and prolonged outages.
*   **Suboptimal Cost Management:** Lack of insights into energy consumption patterns and tariff costs, hindering cost optimization efforts.

This project addresses these challenges by creating a unified data view that supports comprehensive monitoring and analytical capabilities, empowering operators to make data-driven decisions.

### Datasets Involved

The pipeline processes nine core datasets, each providing a crucial piece of the operational puzzle:

*   **`plant_master.csv`**: Contains static information about each energy plant, including ID, name, region, installed capacity, and operator.
*   **`equipment_master.csv`**: Details individual equipment units within each plant, such as equipment ID, type, manufacturer, rated capacity, installation date, and status.
*   **`energy_generation.csv`**: Records timestamped energy generation data, including `Energy_Generated_kWh`, `Performance_Ratio`, `Inverter_Efficiency`, and `Capacity_Factor` for each equipment unit.
*   **`load_consumption.csv`**: Tracks timestamped energy consumption data per region, including `Load_kW`, `Peak_Demand_kW`, `Energy_Consumed_kWh`, `Voltage`, `Current`, `Power_Factor`, and `Load_Factor`.
*   **`fault_events.csv`**: Logs fault occurrences, providing `Fault_ID`, `Timestamp`, `Fault_Type`, `Severity`, `Response_Time_Min`, `Resolution_Time`, `Downtime_Min`, `Energy_Loss_kWh`, and `Cost_Loss_USD`.
*   **`maintenance_logs.csv`**: Contains records of maintenance activities, including `Maintenance_ID`, `Maintenance_Date`, `Maintenance_Type`, `Action_Taken`, `Downtime_Min`, `Maintenance_Cost_USD`, and `Technician_Team`.
*   **`weather.csv`**: Provides timestamped weather data for different regions, such as `Ambient_Temperature`, `Solar_Irradiance`, `Wind_Speed`, `Humidity`, and `Cloud_Cover`.
*   **`tariff_cost.csv`**: Outlines energy tariff rates and demand charges based on timestamp, region, and tariff type.
*   **`equipment_performance_scores.csv`**: Contains pre-calculated performance metrics and risk levels for each equipment unit, including `Average_Performance_Ratio`, `Average_Inverter_Efficiency`, `Average_Capacity_Factor`, `Fault_Count`, `Performance_Score`, and `Risk_Level`.

### Tools and Technologies

*   **Python**: The core programming language for the ETL pipeline.
*   **pandas**: Utilized for efficient data manipulation, cleaning, and analysis.
*   **NumPy**: Used for numerical operations, particularly within pandas.
*   **Google Colab**: The development environment, providing free access to GPUs and easy sharing.
*   **Google Drive**: Used for securely storing and accessing raw datasets.

### Overall Workflow

The project follows a standard ETL methodology:

1.  **Extract**: Raw data is extracted from CSV files stored in Google Drive.
2.  **Transform (Exploratory Data Quality Checks)**: Initial steps involve checking data shapes, previewing records, identifying missing values, detecting duplicate entries, and inspecting data types across all datasets. This phase ensures data quality and prepares for more advanced transformations.
3.  **Load**: The processed data will eventually be loaded into a structured format (e.g., a database or data warehouse) for further analysis, visualization, and application development (future steps).

This notebook focuses on the **Extract** phase and initial **Transform** (data quality checks) steps, laying the groundwork for a robust intelligence platform.

## 1. Import Libraries

In [3]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

## 2. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Define Base Path

In [5]:
BASE_PATH = "/content/drive/MyDrive/Energy Operations Intelligence Platform/data/raw/"

## 4. Load Raw Datasets

In [6]:
plant_master = pd.read_csv(BASE_PATH + "plant_master.csv")
equipment_master = pd.read_csv(BASE_PATH + "equipment_master.csv")
energy_generation = pd.read_csv(BASE_PATH + "energy_generation.csv")
load_consumption = pd.read_csv(BASE_PATH + "load_consumption.csv")
fault_events = pd.read_csv(BASE_PATH + "fault_events.csv")
maintenance_logs = pd.read_csv(BASE_PATH + "maintenance_logs.csv")
weather = pd.read_csv(BASE_PATH + "weather.csv")
tariff_cost = pd.read_csv(BASE_PATH + "tariff_cost.csv")
equipment_performance = pd.read_csv(BASE_PATH + "equipment_performance_scores.csv")

## 5. Initial Data Overview (Shape Check)

In [7]:
datasets = {
    "Plant Master": plant_master,
    "Equipment Master": equipment_master,
    "Energy Generation": energy_generation,
    "Load Consumption": load_consumption,
    "Fault Events": fault_events,
    "Maintenance Logs": maintenance_logs,
    "Weather": weather,
    "Tariff Cost": tariff_cost,
    "Equipment Performance": equipment_performance
}

for name, df in datasets.items():
    print(f"{name:<30} {df.shape}")

Plant Master                   (20, 5)
Equipment Master               (540, 7)
Energy Generation              (17472500, 7)
Load Consumption               (139780, 9)
Fault Events                   (3000, 11)
Maintenance Logs               (1200, 9)
Weather                        (139780, 7)
Tariff Cost                    (139780, 5)
Equipment Performance          (540, 9)


## 6. Preview Sample Records

In [8]:
for name, df in datasets.items():
    print("="*70)
    print(name)
    print("="*70)
    print(df.head())
    print()

Plant Master
   Plant_ID            Plant_Name       Region  Installed_Capacity_MW  \
0         1     Lahore Solar Farm          KPK                     31   
1         2   Karachi Solar Plant  Balochistan                     62   
2         3  Islamabad PV Station       Punjab                     11   
3         4  Faisalabad Solar Hub          KPK                     97   
4         5     Multan Solar Park          KPK                     39   

     Operator  
0       FESCO  
1       LESCO  
2       LESCO  
3  K-Electric  
4       FESCO  

Equipment Master
  Equipment_ID  Plant_ID Equipment_Type Manufacturer  Rated_Capacity_kW  \
0     EQ-00001         1       Inverter      Siemens                125   
1     EQ-00002         1       Inverter          ABB                 50   
2     EQ-00003         1       Inverter           GE                 75   
3     EQ-00004         1       Inverter      Siemens                 75   
4     EQ-00005         1       Inverter           GE       

## 7. Missing Value Analysis

In [9]:
for name, df in datasets.items():
    print("="*70)
    print(name)
    print("="*70)
    print(df.isnull().sum())
    print()

Plant Master
Plant_ID                 0
Plant_Name               0
Region                   0
Installed_Capacity_MW    0
Operator                 0
dtype: int64

Equipment Master
Equipment_ID         0
Plant_ID             0
Equipment_Type       0
Manufacturer         0
Rated_Capacity_kW    0
Installation_Date    0
Status               0
dtype: int64

Energy Generation
Timestamp               0
Plant_ID                0
Equipment_ID            0
Energy_Generated_kWh    0
Performance_Ratio       0
Inverter_Efficiency     0
Capacity_Factor         0
dtype: int64

Load Consumption
Timestamp              0
Region                 0
Load_kW                0
Peak_Demand_kW         0
Energy_Consumed_kWh    0
Voltage                0
Current                0
Power_Factor           0
Load_Factor            0
dtype: int64

Fault Events
Fault_ID             0
Timestamp            0
Plant_ID             0
Equipment_ID         0
Fault_Type           0
                    ..
Response_Time_Min    0
Re

## 8. Duplicate Check

In [10]:
for name, df in datasets.items():
    print(name)
    print("Duplicate Rows:", df.duplicated().sum())
    print()

Plant Master
Duplicate Rows: 0

Equipment Master
Duplicate Rows: 0

Energy Generation
Duplicate Rows: 0

Load Consumption
Duplicate Rows: 0

Fault Events
Duplicate Rows: 0

Maintenance Logs
Duplicate Rows: 0

Weather
Duplicate Rows: 0

Tariff Cost
Duplicate Rows: 0

Equipment Performance
Duplicate Rows: 0



## 9. Data Type Inspection

In [11]:
for name, df in datasets.items():
    print("="*70)
    print(name)
    print("="*70)
    print(df.dtypes)
    print()

Plant Master
Plant_ID                  int64
Plant_Name               object
Region                   object
Installed_Capacity_MW     int64
Operator                 object
dtype: object

Equipment Master
Equipment_ID         object
Plant_ID              int64
Equipment_Type       object
Manufacturer         object
Rated_Capacity_kW     int64
Installation_Date    object
Status               object
dtype: object

Energy Generation
Timestamp                object
Plant_ID                  int64
Equipment_ID             object
Energy_Generated_kWh    float64
Performance_Ratio       float64
Inverter_Efficiency     float64
Capacity_Factor         float64
dtype: object

Load Consumption
Timestamp               object
Region                  object
Load_kW                float64
Peak_Demand_kW         float64
Energy_Consumed_kWh    float64
Voltage                float64
Current                float64
Power_Factor           float64
Load_Factor            float64
dtype: object

Fault Events
Fau